# Dataset Overlap Analysis Guide

This notebook demonstrates how to use the `OpenLinkTokenOverlapAnalyzer` class to identify matching records between two tokenized datasets.

## Overview

The OpenLinkTokenOverlapAnalyzer helps you:
- Find matching records across two datasets based on encrypted `ot.V1` tokens
- Define flexible matching rules (which token types must match)
- Compare overlap rates using different matching criteria
- Get detailed statistics and matched record pairs by comparing deterministic decrypted values

## Setup

Use the same initiate-exchange config for token generation and overlap analysis.

**Recommended:** Resolve the exchange config and participant private key on the driver with `from_exchange_config(...)`.
**Backward compatibility:** Direct encryption-key construction still works if you already manage raw secrets yourself.


In [ ]:
from pyspark.sql import SparkSession

from openlinktoken_pyspark import OpenLinkTokenOverlapAnalyzer, OpenLinkTokenProcessor

# Create Spark session
spark = (
    SparkSession.builder.appName("OverlapAnalysisExample")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Exchange-config inputs used for both token generation and overlap analysis
EXCHANGE_CONFIG_PATH = "/path/to/initiate-exchange-config.json"
PRIVATE_KEY_PATH = "/path/to/participant-private-key.pem"
PRIVATE_KEY_ENV = None

## Create Sample Datasets

Let's create two sample datasets with overlapping records.

In [ ]:
# Dataset A
dataset_a_data = [
    ("A001", "John", "Doe", "1990-01-15", "Male", "98101", "123-45-6789"),
    ("A002", "Jane", "Smith", "1985-06-20", "Female", "94105", "987-65-4321"),
    ("A003", "Bob", "Johnson", "1978-03-10", "Male", "02134", "456-78-9123"),
    ("A004", "Alice", "Williams", "1992-11-05", "Female", "10001", "321-54-9876"),
    ("A005", "Charlie", "Brown", "1988-08-22", "Male", "60614", "789-12-3456"),
]

dataset_a_df = spark.createDataFrame(
    dataset_a_data, ["RecordId", "FirstName", "LastName", "BirthDate", "Sex", "PostalCode", "SocialSecurityNumber"]
)

print("Dataset A Records:")
dataset_a_df.show()

In [ ]:
# Dataset B (has some overlapping records)
dataset_b_data = [
    ("B101", "John", "Doe", "1990-01-15", "Male", "98101", "123-45-6789"),  # Same as A001
    ("B102", "Jane", "Smith", "1985-06-20", "Female", "94105", "987-65-4321"),  # Same as A002
    ("B103", "David", "Lee", "1995-04-18", "Male", "30303", "654-32-1098"),  # Unique to B
    ("B104", "Emma", "Davis", "1982-12-30", "Female", "90210", "234-56-7890"),  # Unique to B
]

dataset_b_df = spark.createDataFrame(
    dataset_b_data, ["RecordId", "FirstName", "LastName", "BirthDate", "Sex", "PostalCode", "SocialSecurityNumber"]
)

print("Dataset B Records:")
dataset_b_df.show()

## Generate Tokens for Both Datasets

Create both tokenized datasets from the same exchange-config inputs.


In [ ]:
# Initialize token processor
processor = OpenLinkTokenProcessor.from_exchange_config(
    exchange_config_path=EXCHANGE_CONFIG_PATH,
    private_key_path=PRIVATE_KEY_PATH,
    private_key_env=PRIVATE_KEY_ENV,
)

# Generate tokens for Dataset A
dataset_a_tokens = processor.process_dataframe(dataset_a_df)
print("Dataset A Tokens:")
dataset_a_tokens.show(5, truncate=False)

# Confirm encrypted tokens are in ot.V1 format
sample_token_a = dataset_a_tokens.select("Token").first()["Token"]
print(f"Sample token prefix: {sample_token_a[:6]}")

In [ ]:
# Generate tokens for Dataset B
dataset_b_tokens = processor.process_dataframe(dataset_b_df)
print("Dataset B Tokens:")
dataset_b_tokens.show(5, truncate=False)

# Optional sanity check
assert dataset_b_tokens.select("Token").first()["Token"].startswith("olt.V1.")

## Analyze Dataset Overlap

### Method 1: Single Rule Set

Let's find matches requiring T1 and T2 tokens to match with an analyzer built from the same exchange-config inputs.


In [ ]:
# Initialize overlap analyzer from the same exchange-config inputs
analyzer = OpenLinkTokenOverlapAnalyzer.from_exchange_config(
    exchange_config_path=EXCHANGE_CONFIG_PATH,
    private_key_path=PRIVATE_KEY_PATH,
    private_key_env=PRIVATE_KEY_ENV,
)

# Analyze overlap with T1 and T2 matching rules
results = analyzer.analyze_overlap(
    dataset_a_tokens,
    dataset_b_tokens,
    matching_rules=["T1", "T2"],
    dataset1_name="Dataset_A",
    dataset2_name="Dataset_B",
)

# Print summary
analyzer.print_summary(results)

### View Matched Record Pairs

In [ ]:
# Show matched record pairs
print("Matched Record Pairs:")
results["matches"].show()

# Expected: Should show A001<->B101 and A002<->B102

In [ ]:
# Debug: Check if decryption is working in the analyzer
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Test the decrypt method manually
analyzer = OpenLinkTokenOverlapAnalyzer.from_exchange_config(
    exchange_config_path=EXCHANGE_CONFIG_PATH,
    private_key_path=PRIVATE_KEY_PATH,
    private_key_env=PRIVATE_KEY_ENV,
)
test_token = dataset_a_tokens.select("Token").filter("RuleId == 'T1'").first()["Token"]
print(f"Sample Token: {test_token}")
print(f"Decrypted: {analyzer._decrypt_token(test_token)}")

# Check if UDF is working
decrypt_udf = udf(analyzer._decrypt_token, StringType())
dataset_a_tokens.withColumn("Decrypted", decrypt_udf("Token")).select("RuleId", "Token", "Decrypted").show(
    5, truncate=False
)

### Method 2: Compare Multiple Rule Sets

Let's see how overlap changes with different matching criteria.

In [ ]:
# Define different rule sets to compare
rule_sets = [
    ["T1"],  # Least strict: only T1 must match
    ["T1", "T2"],  # Medium: T1 AND T2 must match
    ["T1", "T2", "T3"],  # Strict: T1 AND T2 AND T3 must match
    ["T1", "T2", "T3", "T4"],  # Very strict: all 4 tokens must match
]

# Run comparison
multi_results = analyzer.compare_with_multiple_rules(
    dataset_a_tokens, dataset_b_tokens, rule_sets, dataset1_name="Dataset_A", dataset2_name="Dataset_B"
)

# Display comparison
print("\nOverlap Comparison Across Different Matching Rules:")
print("=" * 70)
for result in multi_results:
    rules_str = ", ".join(result["matching_rules"])
    print(
        f"Rules: {rules_str:20} | "
        f"Matches: {result['matching_records_dataset1']:2} | "
        f"Overlap: {result['overlap_percentage']:5.1f}%"
    )

### Access Detailed Statistics

In [ ]:
# Access detailed statistics from any result
result = multi_results[1]  # T1 + T2 result

print(f"Dataset: {result['dataset1_name']}")
print(f"  Total records: {result['total_records_dataset1']}")
print(f"  Matching records: {result['matching_records_dataset1']}")
print(f"  Unique records: {result['unique_to_dataset1']}")
print()
print(f"Dataset: {result['dataset2_name']}")
print(f"  Total records: {result['total_records_dataset2']}")
print(f"  Matching records: {result['matching_records_dataset2']}")
print(f"  Unique records: {result['unique_to_dataset2']}")

## Use Case: Analyzing Data Sharing Potential

Let's assess whether two datasets have sufficient unique records to justify data sharing.

In [ ]:
# Analyze with strict matching criteria
strict_results = analyzer.analyze_overlap(
    dataset_a_tokens,
    dataset_b_tokens,
    matching_rules=["T1", "T2", "T3"],  # Require 3 token types to match
    dataset1_name="Dataset_A",
    dataset2_name="Dataset_B",
)

# Decision logic
overlap_pct = strict_results["overlap_percentage"]
unique_a = strict_results["unique_to_dataset1"]
unique_b = strict_results["unique_to_dataset2"]

print("\n" + "=" * 70)
print("DATA SHARING ASSESSMENT")
print("=" * 70)
print(f"Overlap: {overlap_pct:.1f}%")
print(f"Unique records in Dataset A: {unique_a}")
print(f"Unique records in Dataset B: {unique_b}")
print()

if overlap_pct < 10:
    print("✓ RECOMMENDATION: High potential for data sharing")
    print("  Minimal overlap suggests complementary populations.")
elif overlap_pct < 30:
    print("✓ RECOMMENDATION: Moderate potential for data sharing")
    print("  Some overlap exists but substantial unique populations in each dataset.")
else:
    print("⚠ RECOMMENDATION: Review data sharing need carefully")
    print("  Significant overlap may reduce value of data sharing.")

## Cleanup

In [ ]:
# Stop Spark session
spark.stop()

## Key Takeaways

1. **Token Format**: Encrypted output uses `olt.V1.<JWE compact serialization>`
2. **Shared Exchange Context**: Use the same exchange-config inputs for token generation and overlap analysis
3. **Matching Rules**: Define which token types must match (ALL must match, not just one)
4. **Multiple Criteria**: Use `compare_with_multiple_rules()` to see how overlap changes with stricter matching
5. **Privacy**: Analysis compares deterministic decrypted values; no PHI is exposed
6. **Results**: Get both statistics and detailed matched record pairs

## Next Steps

- Try with your own datasets
- Experiment with different token rule combinations
- Use custom token definitions (see Custom_Token_Definition_Guide.ipynb)
- Scale to larger datasets using your Spark cluster
- Use direct raw-secret constructors only when maintaining a backward-compatible integration